# NLP Exercises (Part 2)

We have 2 exercises in this section. The exercises are:

4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [2]:
import numpy as np
import spacy
from collections import Counter

class BagOfWords:
    """Basic BoW implementation."""

    def __init__(self):
        self.__nlp = spacy.load("en_core_web_sm")
        self.__bow_list = []

    def fit_transform(self, corpus: list):
        """Transform list of strings into BoW array."""
        tokenized_corpus = []
        vocab_set = set()

        for text in corpus:
            tokens = [token.text.lower() for token in self.__nlp(text) if token.is_alpha]
            tokenized_corpus.append(tokens)
            vocab_set.update(tokens)

        self.__bow_list = sorted(vocab_set)
        word_to_idx = {word: i for i, word in enumerate(self.__bow_list)}

        matrix = np.zeros((len(corpus), len(self.__bow_list)), dtype=int)
        for doc_idx, tokens in enumerate(tokenized_corpus):
            counts = Counter(tokens)
            for word, count in counts.items():
                matrix[doc_idx, word_to_idx[word]] = count

        return matrix

    def get_feature_names(self) -> list:
        """Return words corresponding to columns of matrix."""
        return self.__bow_list


# Test
corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]

vectorizer = BagOfWords()
X = vectorizer.fit_transform(corpus)
print(X)
print(vectorizer.get_feature_names())
print(len(vectorizer.get_feature_names()))

[[0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [1 0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]
['a', 'as', 'bag', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'really', 'see', 'sparse', 'the', 'third', 'this', 'throughout', 'us', 'words', 'you']
31


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [4]:
import nltk
nltk.download('book', quiet=True)

True

In [5]:
from nltk.book import *

wall_street = text7.tokens

import re

tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match,tokens))
    return clean
tokens = cleanup()

def build_ngrams():
    ngrams = []
    for i in range(len(tokens)-N+1):
        ngrams.append(tokens[i:i+N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq  = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1;

    return counts

def next_word(text, N, counts):

    token_seq = SEP.join(text.split()[-(N-1):]);
    choices = counts[token_seq].items();

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight;
        if upto > r: return choice
    assert False # should not reach here

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


In [17]:
import random
import re
from nltk.book import *
def clean_generated(text):

    text = re.sub(r"\s+([.!?])", r"\1", text)

    parts = re.split(r'([.!?]\s*)', text)
    result = []

    make_upper = True

    for element in parts:
        if not element:
            continue

        if make_upper and element[0].isalpha():
            element = element[0].upper() + element[1:]

        result.append(element)

        if re.match(r'[.!?]', element):
            make_upper = True
        else:
            make_upper = False

    return "".join(result).strip()
N = 5
SEP = " "
sentence_count = 5

ngrams = build_ngrams()
counts = ngram_freqs(ngrams)

start_seq = random.choice(list(counts.keys()))
generated = start_seq.lower()

current_sentences = 0
while current_sentences < sentence_count:
    token_seq = SEP.join(generated.split()[-(N-1):])

    if token_seq not in counts:
        token_seq = random.choice(list(counts.keys()))
        generated += SEP + token_seq

    word = next_word(generated, N, counts)
    generated += SEP + word

    if word in ['.', '!', '?']:
        current_sentences += 1

generated = clean_generated(generated)
print(generated)

Units and that its remaining businesses are performing well. Dealers said 0 the U.K. Government decision Tuesday to waive its protective golden share in the auto maker raised prospects of a bidding war with all 0 that implies. If a competitor enters the game for example Mr. Hahn could face the dilemma of paying a premium for Nekoosa or seeing the company fall into the arms of a rival. Given that choice associates of Mr. Hahn and industry observers say 0 he is entering uncharted waters. Says Kathryn McAuley an analyst at First Manhattan Co. This is the greatest acquisition challenge 0 he has faced.
